# Replay “Burning the Midnight Oil: Dolos Had Us Fooled”
Bobby Faber, Azimuth Bouche and Sarosh Thalappil · Team Iron Golem.

Run all cells on a **CPU** runtime. This notebook verifies saved publication evidence; it performs no inference and installs no packages. It needs network access only to clone GitHub. CMU discovered the channel/decoder; Hekkaideka characterized the clock/head repair. Our contribution is the integrated audit and synthesis.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

PUBLICATION_COMMIT = "b00fbd232c0d7ae7b9e23ad4dec05a30258c14bb"
REPO = Path(tempfile.mkdtemp(prefix="dolos-replay-")) / "dolos-red-team"
subprocess.run(["git", "clone", "--quiet", "https://github.com/bubskee/dolos-red-team.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", PUBLICATION_COMMIT], check=True)
resolved = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert resolved == PUBLICATION_COMMIT
print("Publication snapshot:", resolved)
print("Checkout:", REPO)

## Verify Audit 1 and report evidence
The scripts check frozen scientific source/fixture hashes, receiver bundles, predictions, character counts and report reproduction traces. The report check regenerates its derived `reviewer-checks.json` in this temporary checkout.

In [ ]:
for command in (["audit/replay.py", "--verify"], ["report/review_checks.py"]):
    subprocess.run([sys.executable, *command], cwd=REPO, check=True)

In [ ]:
def read_json(relative_path):
    return json.loads((REPO / relative_path).read_text())

for profile in ("case_a", "case_b", "case_c"):
    result = read_json(f"audit/v1/runs/{profile}/evaluation.json")
    print(profile, result["status"],
          "selected", result.get("selected_correct"), "/", result.get("selected_n"),
          "comparison", result.get("control_correct"), "/", result.get("control_n"))

report = read_json("report/reviewer-checks.json")
cmu = next(row for row in report["cmu_reproduction"]
           if row["phase"] == "D1" and row["generator"] == row["scorer"] == "dolos")
assert (cmu["correct"], cmu["fixed_n"], cmu["available_n"], cmu["missing"]) == (337, 384, 370, 14)
print("Separate CMU-style replication:", cmu)
print("Bridge credential dependence retained:", report["bridge"]["retained_js"])

## Audit 2: partial and stopped
The following cell displays the completed Dolos evaluation and recorded priority-screen verification. It does not run the Torch-dependent v2 evaluator. For that independent replay, install `audit/requirements.txt` in a compatible environment and run `audit/v2/verify_run.py --completed-only` plus `audit/v2/verify_priority.py`.

Four mixed-task controls passed positive controls with no candidate. Two pure-tool screens failed positive controls and are inconclusive. Missing full-profile evaluations are not negative results. Numerical scorer checks were planned but not run.

In [ ]:
result = read_json("audit/v2/runs/dolos/evaluation.json")
assert (result["selected_correct"], result["selected_n"], result["control_correct"], result["control_n"]) == (176, 192, 13, 192)
print(json.dumps(result, indent=2))
print("Recorded priority-screen verification:")
print(json.dumps(read_json("audit/v2/priority_verification.json"), indent=2))

## Interpretation
Audit 1 tests eight-character prefixes over six cases and two seeds. Audit 2 remains partial. The separate 32-character replication recovered no complete keys; its denominator and settings differ, so do not pool it with the audits. These studies do not provide broad false-positive calibration. Exact public-sentence coverage is exact-copy compliance, not semantic factuality.

For fresh synthetic-canary inference, read `audit/README.md` in the checkout. That workflow downloads pinned model assets and requires compatible CUDA compute. This notebook does not start it.